[![Open In Colab](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-colab.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/structural-break/quickstarters/baseline/baseline.ipynb)
[![Open In Kaggle](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-kaggle.svg)](https://www.kaggle.com/code/crunchdao/structural-break-baseline)

In [1]:
# Install the Crunch CLI
%pip install --upgrade crunch-cli

!crunch setup-notebook structural-break QZF30iMIUL21v4pmevqmXJg8

Note: you may need to restart the kernel to use updated packages.
crunch-cli, version 7.5.0

---
Your token seems to have expired or is invalid.

Please follow this link to copy and paste your new setup command:
https://hub.crunchdao.com/competitions/structural-break/submit

If you think that is an error, please contact an administrator.


# Your model

## Setup

In [2]:
import os
import typing

# Import your dependencies
import joblib
import pandas as pd
import scipy
import sklearn.metrics
import numpy as np
import pandas as pd
import scipy.stats as st


from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone as sk_clone
from sklearn.pipeline import Pipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import SGDRegressor

import traceback
import warnings
import sys

from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression

from scipy.stats import wasserstein_distance, energy_distance
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
import crunch

# Load the Crunch Toolings
crunch = crunch.load_notebook()

loaded inline runner with module: <module '__main__'>

cli version: 7.5.0
available ram: 30.51 gb
available cpu: 16 core
----


## Understanding the Data

The dataset consists of univariate time series, each containing ~2,000-5,000 values with a designated boundary point. For each time series, you need to determine whether a structural break occurred at this boundary point.

The data was downloaded when you setup your local environment and is now available in the `data/` directory.

In [4]:
X_train, y_train, X_test = crunch.load_data()

data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_train.parquet (204327238 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_test.reduced.parquet (2380918 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_train.parquet (61003 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_test.reduced.parquet (2655 bytes)
data/y_test.reduced.parquet: already exists, file length match


In [5]:
import pandas as pd

data_dir = "/home/endrem/project/structural_break/competitions/competitions/structural-break/data"

# Train
X_train = pd.read_parquet(f"{data_dir}/X_train.parquet")
y_train = pd.read_parquet(f"{data_dir}/y_train.parquet")["structural_breakpoint"]

# Test (group into a list of DataFrames by dataset id, like the platform)
_x_test_df = pd.read_parquet(f"{data_dir}/X_test.reduced.parquet")
X_test = []
for id_val, df in _x_test_df.groupby(_x_test_df.index.get_level_values("id")):
    df.name = id_val
    X_test.append(df)

# Optional: local scoring target for the reduced test
y_test = pd.read_parquet(f"{data_dir}/y_test.reduced.parquet")["structural_breakpoint"]


In [ ]:
def _mad(x):
    if len(x) == 0:
        return 0.0
    m = np.median(x)
    return float(np.median(np.abs(x - m)))

def _acf(x, lag):
    if len(x) <= lag:
        return 0.0
    s = pd.Series(x)
    v = s.autocorr(lag=lag)
    return 0.0 if pd.isna(v) else float(v)

def _logp(p):
    try:
        p = float(p)
        p = np.clip(p, 1e-300, 1.0)
        return -np.log10(p)
    except Exception:
        return 0.0

def _safe_p(fn, *args, **kwargs):
    try:
        res = fn(*args, **kwargs)
        # return p-value if tuple-like (stat, p), or attribute pvalue
        if isinstance(res, tuple) and len(res) >= 2:
            return float(res[1]) if res[1] is not None else 1.0
        if hasattr(res, "pvalue"):
            return float(res.pvalue) if res.pvalue is not None else 1.0
        # some tests return statistic only
        return None
    except Exception:
        return None

def _safe_stat(fn, *args, **kwargs):
    try:
        res = fn(*args, **kwargs)
        if isinstance(res, tuple) and len(res) >= 1:
            return float(res[0]) if res[0] is not None else 0.0
        if hasattr(res, "statistic"):
            return float(res.statistic)
        # return numeric if single value
        return float(res)
    except Exception:
        return 0.0

def _hist_kl(a, b, bins=32):
    # small histogram-based KL (symmetrized)
    if len(a) == 0 or len(b) == 0:
        return 0.0
    try:
        mn = min(a.min(), b.min())
        mx = max(a.max(), b.max())
        if mn == mx:
            return 0.0
        h_a, _ = np.histogram(a, bins=bins, range=(mn, mx), density=True)
        h_b, _ = np.histogram(b, bins=bins, range=(mn, mx), density=True)
        # add tiny eps
        eps = 1e-12
        h_a = h_a + eps
        h_b = h_b + eps
        kl1 = st.entropy(h_a, h_b)
        kl2 = st.entropy(h_b, h_a)
        return float(0.5 * (kl1 + kl2))
    except Exception:
        return 0.0

def _fft_peak(x):
    if len(x) < 3:
        return 0.0, 0.0
    try:
        xf = np.fft.rfft(x - np.mean(x))
        ps = np.abs(xf)**2
        freqs = np.fft.rfftfreq(len(x))
        idx = np.argmax(ps)
        return float(freqs[idx]), float(ps[idx])
    except Exception:
        return 0.0, 0.0
    

# helper functions for new features
def _rms(x):
    if len(x) == 0:
        return 0.0
    return float(np.sqrt(np.mean(np.square(x))))

def _hist_entropy(x, bins=32):
    if len(x) == 0:
        return 0.0
    h, _ = np.histogram(x, bins=bins, density=True)
    h = h + 1e-12
    return float(st.entropy(h))

def _spec_entropy(x):
    # spectral entropy from power spectrum
    if len(x) < 3:
        return 0.0
    xf = np.fft.rfft(x - np.mean(x))
    ps = np.abs(xf) ** 2
    ps = ps + 1e-12
    pnorm = ps / ps.sum()
    return float(-np.sum(pnorm * np.log(pnorm)))

def _bandpower_ratio(x, frac=0.25):
    # ratio low / high band power; frac defines low band as lowest frac of freqs
    if len(x) < 3:
        return 0.0
    xf = np.fft.rfft(x - np.mean(x))
    ps = np.abs(xf) ** 2
    freqs = np.fft.rfftfreq(len(x))
    if freqs.size == 0:
        return 0.0
    cutoff = freqs.max() * frac
    low_mask = freqs <= cutoff
    high_mask = ~low_mask
    low = ps[low_mask].sum() if low_mask.any() else 0.0
    high = ps[high_mask].sum() if high_mask.any() else 0.0
    return float((low + 1e-12) / (high + 1e-12))

def _mean_abs_change(x):
    if len(x) < 2:
        return 0.0
    return float(np.mean(np.abs(np.diff(x))))

def _median_abs_change(x):
    if len(x) < 2:
        return 0.0
    return float(np.median(np.abs(np.diff(x))))

def _slope(x):
    # fit linear slope over sample index
    n = len(x)
    if n < 2:
        return 0.0
    t = np.arange(n, dtype=float)
    try:
        m = np.polyfit(t, x, 1)[0]
        return float(m)
    except Exception:
        return 0.0

def _cohens_d(a, b):
    if len(a) + len(b) == 0:
        return 0.0
    ma, mb = np.mean(a) if len(a) else 0.0, np.mean(b) if len(b) else 0.0
    va = np.var(a, ddof=1) if len(a) > 1 else 0.0
    vb = np.var(b, ddof=1) if len(b) > 1 else 0.0
    pooled = np.sqrt(((len(a)-1)*va + (len(b)-1)*vb) / max(1, (len(a)+len(b)-2)))
    if pooled < 1e-12:
        return 0.0
    return float((mb - ma) / pooled)
    

def _fe_one(u: pd.DataFrame) -> np.ndarray:
    a = u["value"][u["period"] == 0].to_numpy(dtype=float)
    b = u["value"][u["period"] == 1].to_numpy(dtype=float)
    eps = 1e-12

    # basic counts
    len_a = a.size
    len_b = b.size

    # robust stats
    mean_a, mean_b = (float(a.mean()) if len_a else 0.0), (float(b.mean()) if len_b else 0.0)
    med_a, med_b = (float(np.median(a)) if len_a else 0.0), (float(np.median(b)) if len_b else 0.0)
    std_a = float(a.std(ddof=1)) if len_a > 1 else 0.0
    std_b = float(b.std(ddof=1)) if len_b > 1 else 0.0
    mad_a, mad_b = _mad(a), _mad(b)
    iqr_a = float(np.subtract(*np.percentile(a, [75, 25]))) if len_a else 0.0
    iqr_b = float(np.subtract(*np.percentile(b, [75, 25]))) if len_b else 0.0
    trim_a = float(st.trim_mean(a, 0.1)) if len_a else 0.0
    trim_b = float(st.trim_mean(b, 0.1)) if len_b else 0.0
    p90_a, p10_a = (float(np.percentile(a, 90)) if len_a else 0.0), (float(np.percentile(a, 10)) if len_a else 0.0)
    p90_b, p10_b = (float(np.percentile(b, 90)) if len_b else 0.0), (float(np.percentile(b, 10)) if len_b else 0.0)
    nz_a = float((a != 0).sum()) / (len_a + eps)
    nz_b = float((b != 0).sum()) / (len_b + eps)
    skew_a = float(st.skew(a)) if len_a > 2 else 0.0
    skew_b = float(st.skew(b)) if len_b > 2 else 0.0
    kur_a = float(st.kurtosis(a)) if len_a > 3 else 0.0
    kur_b = float(st.kurtosis(b)) if len_b > 3 else 0.0

    # acf lags
    acf_diffs = [_acf(a, lag) - _acf(b, lag) for lag in range(1, 6)]

    # fft peaks
    f_a, p_a = _fft_peak(a) if len_a else (0.0, 0.0)
    f_b, p_b = _fft_peak(b) if len_b else (0.0, 0.0)

    # statistical tests - p-values when available
    p_t = _safe_p(st.ttest_ind, a, b, equal_var=False) or 1.0
    p_ks = _safe_p(st.ks_2samp, a, b, alternative="two-sided", mode="asymp") or 1.0
    p_mw = _safe_p(st.mannwhitneyu, a, b, alternative="two-sided") or 1.0
    p_lv = _safe_p(st.levene, a, b, center="median") or 1.0
    p_fl = _safe_p(st.fligner, a, b) or 1.0
    p_ba = _safe_p(st.bartlett, a, b) or 1.0
    p_med = _safe_p(st.median_test, a, b) or 1.0

    # Cramer-von Mises (statistic) if available
    try:
        cvm_stat = _safe_stat(st.cramervonmises_2samp, a, b)
    except Exception:
        cvm_stat = _safe_stat(lambda x, y: (0.0,), a, b)

    # permutation test (approx) - may not exist in older scipy, fallback None
    perm_p = None
    try:
        if hasattr(st, "permutation_test"):
            res = st.permutation_test((a, b), lambda xa, xb: np.mean(xa) - np.mean(xb), alternative="two-sided", permutation_type="independent", vectorized=False, n_resamples=199)
            perm_p = float(res.pvalue)
    except Exception:
        perm_p = None

    # distances
    try:
        wdist = float(wasserstein_distance(a, b)) if len_a and len_b else 0.0
    except Exception:
        wdist = 0.0
    try:
        edist = float(energy_distance(a, b)) if len_a and len_b else 0.0
    except Exception:
        edist = 0.0
    kl = _hist_kl(a, b)

    # aggregated neg-log10 mean of available p-values
    pvals = [p for p in [p_t, p_ks, p_mw, p_lv, p_fl, p_ba, p_med, perm_p] if p is not None]
    neglog_mean = float(np.mean([_logp(p) for p in pvals])) if len(pvals) else 0.0


    feats = [
        # basic
        (mean_b - mean_a),
        abs(mean_b - mean_a),
        (med_b - med_a),
        (std_b + eps) / (std_a + eps),
        (mad_b + eps) / (mad_a + eps),
        (iqr_b + eps) / (iqr_a + eps),
        (trim_b - trim_a),
        (p90_b - p90_a),
        (p10_b - p10_a),
        (p90_b - p10_b) - (p90_a - p10_a),

        # counts
        len_a, len_b, (len_b + eps) / (len_a + eps), (nz_b - nz_a),

        # shape
        (skew_b - skew_a),
        (kur_b - kur_a),

        # acf diffs
        *acf_diffs,

        # fft
        (f_b - f_a), ( (p_b + eps) / (p_a + eps) ),

        # pvals (as neglog10)
        _logp(p_t), _logp(p_ks), _logp(p_mw), _logp(p_lv), _logp(p_fl), _logp(p_ba), _logp(p_med),

        # cvm stat and perm p
        float(cvm_stat), (perm_p if perm_p is not None else 1.0),

        # distances
        wdist, edist, kl,

        # aggregate
        neglog_mean,
    ]

    # new derived features
    slope_a, slope_b = _slope(a), _slope(b)
    mean_abs_change_a, mean_abs_change_b = _mean_abs_change(a), _mean_abs_change(b)
    median_abs_change_a, median_abs_change_b = _median_abs_change(a), _median_abs_change(b)
    max_a, max_b = (float(np.max(a)) if a.size else 0.0), (float(np.max(b)) if b.size else 0.0)
    min_a, min_b = (float(np.min(a)) if a.size else 0.0), (float(np.min(b)) if b.size else 0.0)
    range_a = max_a - min_a
    range_b = max_b - min_b
    rms_a, rms_b = _rms(a), _rms(b)
    ent_a, ent_b = _hist_entropy(a), _hist_entropy(b)
    sent_a, sent_b = _spec_entropy(a), _spec_entropy(b)
    bp_a, bp_b = _bandpower_ratio(a), _bandpower_ratio(b)
    abs_cum_a = float(np.sum(np.abs(np.diff(a)))) if a.size > 1 else 0.0
    abs_cum_b = float(np.sum(np.abs(np.diff(b)))) if b.size > 1 else 0.0
    cohens = _cohens_d(a, b)

    # append new features to feats list
    feats.extend([
        (slope_b - slope_a),
        (mean_abs_change_b - mean_abs_change_a),
        (median_abs_change_b - median_abs_change_a),
        (max_b - max_a),
        (min_b - min_a),
        ((range_b + eps) / (range_a + eps)),
        ((rms_b + eps) / (rms_a + eps)),
        (ent_b - ent_a),
        (sent_b - sent_a),
        ((bp_b + eps) / (bp_a + eps)),
        (abs_cum_b - abs_cum_a),
        cohens,
    ])

    # Initialize MSAR-specific features
    msar_features = [0.0] * 6  # Placeholder for 6 MSAR features

    try:
        # Fit MSAR models for both periods
        model_a = MarkovAutoregression(a, k_regimes=2, order=1).fit(disp=False)
        model_b = MarkovAutoregression(b, k_regimes=2, order=1).fit(disp=False)

        # Extract regime-specific means
        mean_regime1_a = model_a.smoothed_marginal_probabilities[0].mean()
        mean_regime2_a = model_a.smoothed_marginal_probabilities[1].mean()
        mean_regime1_b = model_b.smoothed_marginal_probabilities[0].mean()
        mean_regime2_b = model_b.smoothed_marginal_probabilities[1].mean()

        # Extract transition probabilities
        trans_prob_1_to_2_a = model_a.regime_transition[0, 1]
        trans_prob_2_to_1_a = model_a.regime_transition[1, 0]
        trans_prob_1_to_2_b = model_b.regime_transition[0, 1]
        trans_prob_2_to_1_b = model_b.regime_transition[1, 0]

        # Extract average duration in regimes
        duration_regime1_a = np.mean(np.diff(np.where(np.diff(model_a.smoothed_marginal_probabilities.argmax(axis=0)) != 0)))
        duration_regime2_a = np.mean(np.diff(np.where(np.diff(model_a.smoothed_marginal_probabilities.argmax(axis=0)) != 0)))
        duration_regime1_b = np.mean(np.diff(np.where(np.diff(model_b.smoothed_marginal_probabilities.argmax(axis=0)) != 0)))
        duration_regime2_b = np.mean(np.diff(np.where(np.diff(model_b.smoothed_marginal_probabilities.argmax(axis=0)) != 0)))

        # Compute MSAR-specific features (differences between periods)
        msar_features = [
            mean_regime1_b - mean_regime1_a,
            mean_regime2_b - mean_regime2_a,
            trans_prob_1_to_2_b - trans_prob_1_to_2_a,
            trans_prob_2_to_1_b - trans_prob_2_to_1_a,
            duration_regime1_b - duration_regime1_a,
            duration_regime2_b - duration_regime2_a,
        ]
    except Exception as e:
        print(f"MSAR feature extraction failed: {e}")

    feats.extend(msar_features)

    # sanitize
    feats = [0.0 if (isinstance(v, float) and (np.isnan(v) or np.isinf(v))) else float(v) for v in feats]
    return np.asarray(feats, dtype=float)

def _fe_table(X_train: pd.DataFrame, FEATURES_LIST) -> pd.DataFrame:
    rows, ids = [], []
    for gid, gdf in X_train.groupby(level="id"):
        fe_one = _fe_one(gdf)
        rows.append(fe_one)
        ids.append(gid)
    return pd.DataFrame(rows, index=pd.Index(ids, name="id"), columns=FEATURES_LIST)


FEATURES_TEST = [
    # basic moments / robust stats
    "mean_diff", "abs_mean_diff", "median_diff",
    "std_ratio", "mad_ratio", "iqr_ratio", "trim_mean_diff",
    "pct90_diff", "pct10_diff", "pct90_10_diff",
    "len_a", "len_b", "len_ratio", "pct_nonzero_diff",

    "skew_diff", "kurt_diff", # shape
    # autocorrelations
    "acf1_diff", "acf2_diff", "acf3_diff", "acf4_diff", "acf5_diff",
    "fft_peak_freq_diff", "fft_peak_power_ratio",# spectral
    # classic tests (log p-values where p exists)
    "p_t_welch", "p_ks", "p_mw", "p_levene", "p_fligner", "p_bartlett", "p_median",
    # Cramer-von Mises (stat) and permutation placeholder
    "cvmd_stat", "perm_p",
    "wasserstein", "energy", "kl_div",# distance metrics / divergences
    "neglog10_p_mean",# aggregated test blend

    # --- additional features (new)
    "slope_diff",               # linear trend slope difference (after - before)
    "mean_abs_change_diff",     # mean of absolute successive changes diff
    "median_abs_change_diff",   # median of absolute successive changes diff
    "max_diff",                 # max(after)-max(before)
    "min_diff",                 # min(after)-min(before)
    "range_diff",               # range(after)/range(before)
    "rms_ratio",                # RMS(after)/RMS(before)
    "entropy_diff",             # histogram entropy difference
    "spec_entropy_diff",        # spectral entropy difference
    "bandpower_lh_ratio",       # low/high band power ratio (after / before)
    "abs_cum_change_diff",      # absolute cumulative change diff
    "cohens_d",                 # standardized mean difference (effect size)
    "msar_mean_regime1",        # Mean of regime 1
    "msar_mean_regime2",        # Mean of regime 2
    "msar_trans_prob_1_to_2",   # Transition probability from regime 1 to 2
    "msar_trans_prob_2_to_1",   # Transition probability from regime 2 to 1
    "msar_duration_regime1",    # Average duration in regime 1
    "msar_duration_regime2",    # Average duration in regime 2
]


F = _fe_table(X_train.loc[[0, 1]], FEATURES_TEST)

TypeError: unhashable type: 'list'

In [11]:
X_train.xs(0).head()

,value,period
time,,
0,-0.005564,0
1,0.003705,0
2,0.013164,0
3,0.007151,0
4,-0.009979,0


In [7]:
def train(X_train: pd.DataFrame, y_train: pd.Series, model_directory_path: str):
    FEATURES_TEST = [
        # basic moments / robust stats
        "mean_diff", "abs_mean_diff", "median_diff",
        "std_ratio", "mad_ratio", "iqr_ratio", "trim_mean_diff",
        "pct90_diff", "pct10_diff", "pct90_10_diff",
        "len_a", "len_b", "len_ratio", "pct_nonzero_diff",

        "skew_diff", "kurt_diff", # shape
        # autocorrelations
        "acf1_diff", "acf2_diff", "acf3_diff", "acf4_diff", "acf5_diff",
        "fft_peak_freq_diff", "fft_peak_power_ratio",# spectral
        # classic tests (log p-values where p exists)
        "p_t_welch", "p_ks", "p_mw", "p_levene", "p_fligner", "p_bartlett", "p_median",
        # Cramer-von Mises (stat) and permutation placeholder
        "cvmd_stat", "perm_p",
        "wasserstein", "energy", "kl_div",# distance metrics / divergences
        "neglog10_p_mean",# aggregated test blend

        # --- additional features (new)
        "slope_diff",               # linear trend slope difference (after - before)
        "mean_abs_change_diff",     # mean of absolute successive changes diff
        "median_abs_change_diff",   # median of absolute successive changes diff
        "max_diff",                 # max(after)-max(before)
        "min_diff",                 # min(after)-min(before)
        "range_diff",               # range(after)/range(before)
        "rms_ratio",                # RMS(after)/RMS(before)
        "entropy_diff",             # histogram entropy difference
        "spec_entropy_diff",        # spectral entropy difference
        "bandpower_lh_ratio",       # low/high band power ratio (after / before)
        "abs_cum_change_diff",      # absolute cumulative change diff
        "cohens_d",                 # standardized mean difference (effect size)
        "msar_mean_regime1",        # Mean of regime 1
        "msar_mean_regime2",        # Mean of regime 2
        "msar_trans_prob_1_to_2",   # Transition probability from regime 1 to 2
        "msar_trans_prob_2_to_1",   # Transition probability from regime 2 to 1
        "msar_duration_regime1",    # Average duration in regime 1
        "msar_duration_regime2",    # Average duration in regime 2
    ]
    # build feature table (index aligned with y_train)
    F = _fe_table(X_train, FEATURES_TEST).reindex(y_train.index).fillna(0.0)
    #F = pd.read_parquet("features.parquet")
    cols_to_drop = ['perm_p',  'neglog10_p_mean'] #['abs_mean_diff', 'len_ratio', 'acf3_diff', 'acf4_diff', 'cvmd_stat', 'pct10_diff', 'wasserstein', 'pct_nonzero_diff']
    cols_present = [c for c in cols_to_drop if c in F.columns]
    cols_indices = [F.columns.get_loc(c) for c in cols_present]
    #cols_indices = [i for i in range(31, 36)]
    cols_to_drop = F.columns[cols_indices]
    print("Indices to drop:", cols_indices, " name:", cols_to_drop.tolist())
    F = F.drop(columns=cols_to_drop, errors='ignore')
    #F = F.iloc[:, :32].copy()
        
    X = F.values
    y = y_train.astype(int).to_numpy()

    base_clfs = {
        #  "lr": Pipeline([
        #      ("scaler", StandardScaler()),
        #      ("clf", LogisticRegression(
        #          max_iter=990,
        #          #solver="saga",
        #          tol=1e-4,
        #          class_weight="balanced",
        #          random_state=42,
        #          n_jobs=-1
        #      ))
        #  ]),        
         #{'class_weight': 'balanced_subsample', 'max_depth': 30, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 300}
        "rf": RandomForestClassifier(
            max_depth= 30, 
            min_samples_leaf= 4, 
            min_samples_split= 2, 
            n_estimators=300, 
            n_jobs=-1, 
            class_weight="balanced_subsample", 
            random_state=42),
        "hgb": HistGradientBoostingClassifier(max_iter=200, random_state=42),
        # "svm_linear": Pipeline([
        #     ("scaler", StandardScaler()),
        #     ("clf", LinearSVC(
        #         class_weight="balanced",
        #         max_iter=1000,
        #         random_state=42
        #     ))
        # ]),
        # "svm_rbf": Pipeline([
        #     ("scaler", StandardScaler()),
        #     ("clf", SVC(
        #         kernel="rbf",
        #         class_weight="balanced",
        #         C=1.0,
        #         random_state=42
        #     ))
        # ]),
        # "knn": Pipeline([
        #     ("scaler", StandardScaler()),
        #     ("clf", KNeighborsClassifier(
        #         n_neighbors=5
        #     ))
        # ]),
        # "nb": GaussianNB(),
        "mlp": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(64),
                max_iter=500,
                early_stopping=True,
                n_iter_no_change=20,
                learning_rate_init=0.0001,                
                random_state=42
            ))
        ]),
        "lgbm": LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=7,
            random_state=42,
            verbose=-1,
        ),
        "xgb": XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=7,
            random_state=42,
            verbosity=1,  # Set verbosity level (0 = silent, 1 = warning, 2 = info)
            #use_label_encoder=False,  # Disable label encoding warning
            eval_metric="logloss"  # Evaluation metric
        ),
    }

    # OOF stacking
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds = {k: np.zeros(len(y), dtype=float) for k in base_clfs}
    avgs = {k: 0.0 for k in base_clfs.keys()}
    for train_idx, val_idx in skf.split(X, y):
        Xt, Xv = X[train_idx], X[val_idx]
        yt, yv = y[train_idx], y[val_idx]
        for name, clf in base_clfs.items():
            # train fresh clone for each fold
            clf_clone = sk_clone(clf)  # proper unfitted clone
            clf_clone.fit(Xt, yt)
            # ensure predict_proba exists; fallback to decision_function + sigmoid
            if hasattr(clf_clone, "predict_proba"):
                if name == "lgbm":
                    Xv = pd.DataFrame(Xv, columns=F.columns.tolist())
                    
                val_pred = clf_clone.predict_proba(Xv) # expected shape (1, n_classes) 
                #print("val_pred.shape",val_pred.shape)
                #print("val_pred", val_pred[0].shape)
                val_pred = val_pred[:, 1]
                oof_preds[name][val_idx] = val_pred
                val_score = sklearn.metrics.roc_auc_score(yv, oof_preds[name][val_idx]) #target, pred
                #print(f"  {name}: {val_score:.2f}")
                avgs[name] += val_score / n_splits
            elif hasattr(clf_clone, "decision_function"):
                oof_preds[name][val_idx] = _sigmoid(clf_clone.decision_function(Xv))
            else:  # For regressors like SGDRegressor
                val_pred = clf_clone.predict(Xv)
                oof_preds[name][val_idx] = val_pred

    print("OOF AUC scores:")
    for name, score in avgs.items():
        print(f"  {name}: {score:.2f}")
        
    # meta training on OOF preds
    OOF = np.vstack([oof_preds[name] for name in base_clfs]).T
    print("OOF min max:", OOF.min(), OOF.max()) 
    meta_scaler = StandardScaler()
    OOF_scaled = meta_scaler.fit_transform(OOF)
    meta = LogisticRegression(max_iter=990, class_weight="balanced")
    #meta = LogisticRegression(max_iter=990, solver="lbfgs", tol=1e-4, class_weight="balanced", C=0.5, random_state=42, n_jobs=-1)

    meta.fit(pd.DataFrame(OOF_scaled, columns=base_clfs.keys()), y) #OOF_scaled

    # retrain base learners on full data
    for name, clf in base_clfs.items():
        clf.fit(X, y)

    # save bundle
    bundle = {
        "base": base_clfs,
        "meta": meta,
        "meta_scaler": meta_scaler,
        "features": F.columns.tolist(),  # should match FEATURES_TEST order
    }
    os.makedirs(model_directory_path, exist_ok=True)
    joblib.dump(bundle, os.path.join(model_directory_path, "model.joblib"))

def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

def infer(X_test: typing.Iterable[pd.DataFrame], model_directory_path: str):
    bundle = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    base = bundle["base"]
    meta = bundle.get("meta", None)
    meta_scaler = bundle.get("meta_scaler", None)
    feature_names = bundle["features"]

    yield  # ready

    for dataset in X_test:
        f = _fe_one(dataset).reshape(1, -1)  # same order as training
        indices_to_ignore = [31,35] #[1, 12, 18, 19, 30, 8, 32, 13]
        #indices_to_ignore = [i for i in range(31, 36)] #f.shape[1]
        #print("indices_to_ignore:", indices_to_ignore)
        n_features = f.shape[1]
        keep_idxs = [i for i in range(n_features) if i not in set(indices_to_ignore)]
        f = f[:, keep_idxs]  
        
        #f = f[:,:32].copy()
        
        # If _fe_one or feature order mismatch occurs, you can compute a DataFrame and reindex.
        # e.g., df = pd.DataFrame([f], columns=feature_names) ; arr = df[feature_names].values

        base_preds = []
        for name, clf in base.items():
            if isinstance(clf, Pipeline):
                p = float(clf.predict_proba(f)[0, 1])
            else:
                #print("name:",name)
                if name == "lgbm":
                    f = pd.DataFrame(f, columns=feature_names)
                if hasattr(clf, "predict_proba"):
                    p = float(clf.predict_proba(f)[0, 1])
                elif hasattr(clf, "decision_function"):
                    p = float(_sigmoid(clf.decision_function(f)))
                else:  # For regressors like SGDRegressor
                    p = float(_sigmoid(clf.predict(f)))
            base_preds.append(p)

        stacked = np.array(base_preds).reshape(1, -1)
        if meta is not None: # and meta_scaler is not None:
            stacked_scaled = meta_scaler.transform(stacked) #stacked #
            #p_final = float(meta.predict_proba(stacked_scaled)[0, 1]) #warn
            p_final = float(meta.predict_proba(pd.DataFrame(stacked_scaled, columns=base.keys()))[0, 1]) #warn
            
        else:
            p_final = float(np.mean(stacked)) # fallback: average

        # guarantee [0,1] and deterministic float
        p_final = float(np.clip(p_final, 0.0, 1.0))
        yield p_final

model_dir = "model"
train(X_train, y_train, model_dir)

#print("infer")
#gen = infer(X_test, model_dir)


/tmp/ipykernel_8048/3108933755.py:341: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  feats = [0.0 if (isinstance(v, float) and (np.isnan(v) or np.isinf(v))) else float(v) for v in feats]
/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/tmp/ipykernel_8048/3108933755.py:

MSAR feature extraction failed: SVD did not converge


/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/tmp/ipykernel_8048/3108933755.py:341: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  feats = [0.0 if (isinstance(v, float) and (np.isnan(v) or np.isinf(v))) else float(v) for v in feats]
/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/endrem/project/structural_br

MSAR feature extraction failed: SVD did not converge


/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/tmp/ipykernel_8048/3108933755.py:341: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  feats = [0.0 if (isinstance(v, float) and (np.isnan(v) or np.isinf(v))) else float(v) for v in feats]
/home/endrem/project/structural_br

MSAR feature extraction failed: SVD did not converge


/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/tmp/ipykernel_8048/3108933755.py:341: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  feats = [0.0 if (isinstance(v, float) and (np.isnan(v) or np.isinf(v))) else float(v) for v in feats]
/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/tmp/ipykernel_8048/3108933755.py:

In [19]:
from sklearn.model_selection import GridSearchCV
base_clfs = {
        "lr": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=990,
                #solver="saga",
                tol=1e-4,
                class_weight="balanced",
                random_state=42,
                n_jobs=1
            ))
        ]),        
    "rf": RandomForestClassifier(n_estimators=200, n_jobs=-1, class_weight="balanced", random_state=42),
    "hgb": HistGradientBoostingClassifier(max_iter=200, random_state=42),
}

param_grid = {
    #{'clf__C': 100, 'clf__solver': 'liblinear'}
    "clf__C": [0.01, 0.1, 1, 10, 100],
    "clf__solver": ["liblinear", "saga"],
}
grid_search = GridSearchCV(base_clfs["lr"], param_grid, cv=5, scoring="roc_auc", n_jobs=1)
F = pd.read_parquet("features.parquet", engine="fastparquet")
X = F.values
y = y_train.astype(int).to_numpy()
grid_search.fit(X, y)
print(grid_search.best_params_)

/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
 

{'clf__C': 100, 'clf__solver': 'liblinear'}


In [21]:
F.columns

NameError: name 'F' is not defined

In [22]:
crunch.test(
    # force_first_train=False,
    no_determinism_check=False,
)

15:37:06 no forbidden library found
15:37:06 
15:37:06 started
15:37:06 running local test
15:37:06 internet access isn't restricted, no check will be done
15:37:06 
15:37:06 starting unstructured loop...
15:37:06 executing - command=train


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_train.parquet (204327238 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_test.reduced.parquet (2380918 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_train.parquet (61003 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_test.reduced.parquet (2655 bytes)
data/y_test.reduced.parquet: already exists, file length match
Indices to drop: [31, 35]  name: ['perm_p', 'neglog10_p_mean']
OOF AUC scores:
  rf: 0.66
  hgb: 0.64
  mlp: 0.60
  l

15:38:03 executing - command=infer
/tmp/ipykernel_16895/3108933755.py:341: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  feats = [0.0 if (isinstance(v, float) and (np.isnan(v) or np.isinf(v))) else float(v) for v in feats]
Traceback (most recent call last):
  File "https://github.com/crunchdao/competitions/raw/refs/heads/master/competitions/structural-break/scoring/runner.py", line 31, in run
  File "/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/crunch/runner/local.py", line 559, in execute
    result = utils.smart_call(
             ^^^^^^^^^^^^^^^^^
  File "/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/crunch/utils.py", line 270, in smart_call
    return function(**a

SystemExit: 1

/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [53]:
prediction = pd.read_parquet("data/prediction.parquet")
prediction

,prediction
id,
10001,0.305937
10002,0.279738
10003,0.622980
10004,0.076988
10005,0.160581
...,...
10097,0.188854
10098,0.158334
10099,0.186671


In [87]:
FEATURES_TEST = [
    # basic moments / robust stats
    "mean_diff", "abs_mean_diff", "median_diff",
    "std_ratio", "mad_ratio", "iqr_ratio", "trim_mean_diff",
    "pct90_diff", "pct10_diff", "pct90_10_diff",
    "len_a", "len_b", "len_ratio", "pct_nonzero_diff",

    "skew_diff", "kurt_diff", # shape
    # autocorrelations
    "acf1_diff", "acf2_diff", "acf3_diff", "acf4_diff", "acf5_diff",
    "fft_peak_freq_diff", "fft_peak_power_ratio",# spectral
    # classic tests (log p-values where p exists)
    "p_t_welch", "p_ks", "p_mw", "p_levene", "p_fligner", "p_bartlett", "p_median",
    # Cramer-von Mises (stat) and permutation placeholder
    "cvmd_stat", "perm_p",
    "wasserstein", "energy", "kl_div",# distance metrics / divergences
    "neglog10_p_mean",# aggregated test blend

        # --- additional features (new)
    "slope_diff",               # linear trend slope difference (after - before)
    "mean_abs_change_diff",     # mean of absolute successive changes diff
    "median_abs_change_diff",   # median of absolute successive changes diff
    "max_diff",                 # max(after)-max(before)
    "min_diff",                 # min(after)-min(before)
    "range_diff",               # range(after)/range(before)
    "rms_ratio",                # RMS(after)/RMS(before)
    "entropy_diff",             # histogram entropy difference
    "spec_entropy_diff",        # spectral entropy difference
    "bandpower_lh_ratio",       # low/high band power ratio (after / before)
    "abs_cum_change_diff",      # absolute cumulative change diff
    "cohens_d",                 # standardized mean difference (effect size)
]
F = _fe_table(X_train, FEATURES_TEST) #.reindex(y_train.index).fillna(0.0)
F.to_parquet("features.parquet", index=True)



In [54]:
target = pd.read_parquet("data/y_test.reduced.parquet")["structural_breakpoint"]

# Call the scoring function
sklearn.metrics.roc_auc_score(
    target,
    prediction,
) # 0.66 all models and 26 feat #0.667 (28 feat add lgbm, rm lr) # 0.64 (selected 28 feat) # 0.639(36) 0.64(30) #0.675 (26 feat), 0.636 (14 feat) # 0.615, 0.6291

0.5929577464788732

In [ ]:
# Validate with proper out‑of‑fold (OOF) ROC AUC by splitting by dataset id (StratifiedKFold on ids). See train and features functions _fe_one / _fe_table.
# Increase model diversity: add tree ensembles + gradient boosting + simple rule models (neglog p blend), then stack OOF predictions with a small meta‑learner. See infer for the inference contract.
# Feature pruning: use permutation importance or SHAP on OOF preds to drop noisy features — fewer stable features often improve AUC.
# Robustify features: winsorize/clip, log/scale skewed values, add more time‑series aggregates (rolling medians, diffs, FFT peaks, PSD ratios, spectral entropy).
# Hyperparameter search: RandomizedSearch on base learners (n_estimators, max_depth, learning_rate). Use class_weight or balanced sampling.
# Ensemble strategy for AUC: stack OOF probs with a LogisticRegression meta (or rank‑average base scores); averaging ranks often improves AUC.
# Determinism/calibration: ensure deterministic seeds, clip outputs to [0,1], optionally calibrate meta with CalibratedClassifierCV.

In [30]:
# python
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import joblib, numpy as np, os
from sklearn.inspection import permutation_importance

def train_stack(F_df, y_series, model_dir):
    X = F_df.values
    y = y_series.astype(int).to_numpy()
    base = {
        "rf": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
        "hgb": HistGradientBoostingClassifier(max_iter=300, random_state=42),
        #"lr_lin": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "lr": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                solver="saga",        # robust, supports l1/l2, and n_jobs
                max_iter=4000,        # increase iterations
                tol=1e-3,             # tighter tolerance
                C=1.0,                # regularization strength (tuneable)
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ]),
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = {k: np.zeros(len(y)) for k in base}
    for tr, va in skf.split(X, y):
        Xt, Xv = X[tr], X[va]
        yt = y[tr]
        for name, clf in base.items():
            clf_cloned = sk_clone(clf)  # proper unfitted clone
            #clf_cloned = joblib.loads(joblib.dumps(clf))
            clf_cloned.fit(Xt, yt)
            preds = clf_cloned.predict_proba(Xv)[:, 1] if hasattr(clf_cloned, "predict_proba") else clf_cloned.decision_function(Xv)
            oof[name][va] = preds
    OOF = np.vstack([oof[n] for n in base]).T
    meta_scaler = StandardScaler().fit(OOF)
    #meta = LogisticRegression(max_iter=1000, class_weight="balanced")
    meta = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="saga",
            max_iter=4000,
            tol=1e-4,
            C=0.5,               # stronger regularization can help convergence & generalization
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])
    meta.fit(meta_scaler.transform(OOF), y)
    # retrain full base
    for n, c in base.items():
        c.fit(X, y)
    joblib.dump({"base": base, "meta": meta, "meta_scaler": meta_scaler, "features": F_df.columns.tolist()}, os.path.join(model_dir, "model.joblib"))

    X_scaled = StandardScaler().fit(F_df.values).transform(F_df.values)
    res_lr = permutation_importance(base["lr"], X_scaled, y, n_repeats=36, random_state=42, scoring="roc_auc", n_jobs=-1)
    res_rf = permutation_importance(base["rf"], X_scaled, y, n_repeats=36, random_state=42, scoring="roc_auc", n_jobs=-1)
    res_hgb = permutation_importance(base["hgb"], X_scaled, y, n_repeats=36, random_state=42, scoring="roc_auc", n_jobs=-1)

    cols = F_df.columns.tolist()
    imp_lr = dict(zip(cols, res_lr.importances_mean))
    imp_rf = dict(zip(cols, res_rf.importances_mean))
    imp_hgb = dict(zip(cols, res_hgb.importances_mean))

    # sorted feature lists by importance for each model
    sorted_lr = sorted(imp_lr, key=imp_lr.get, reverse=True)
    sorted_rf = sorted(imp_rf, key=imp_rf.get, reverse=True)
    sorted_hgb = sorted(imp_hgb, key=imp_hgb.get, reverse=True)

    # average the three importance arrays and produce final ranking
    avg_vals = (res_lr.importances_mean + res_rf.importances_mean + res_hgb.importances_mean) / 3.0
    imp_avg = dict(zip(cols, avg_vals))
    sorted_avg = sorted(imp_avg, key=imp_avg.get, reverse=True)

    print("num columns:", len(cols))
    print("top avg features:", sorted_avg[:36]) 
    return top 

####
# python

F = pd.read_parquet("features.parquet")

X = F.values
top = train_stack(F, y_train, "model_stack")
print("Top features:", top)




num columns: 36
top avg features: ['kl_div', 'p_t_welch', 'neglog10_p_mean', 'mad_ratio', 'iqr_ratio', 'acf1_diff', 'trim_mean_diff', 'acf5_diff', 'pct90_diff', 'fft_peak_freq_diff', 'pct90_10_diff', 'acf3_diff', 'len_ratio', 'perm_p', 'acf4_diff', 'skew_diff', 'kurt_diff', 'fft_peak_power_ratio', 'len_a', 'len_b', 'std_ratio', 'mean_diff', 'cvmd_stat', 'median_diff', 'acf2_diff', 'p_mw', 'pct10_diff', 'p_median', 'pct_nonzero_diff', 'energy', 'p_fligner', 'p_bartlett', 'p_levene', 'abs_mean_diff', 'wasserstein', 'p_ks']
Top features: ['trim_mean_diff', 'median_diff', 'pct90_diff', 'acf2_diff', 'energy', 'perm_p', 'kl_div', 'pct90_10_diff', 'mad_ratio', 'acf5_diff', 'p_ks', 'mean_diff', 'acf1_diff', 'iqr_ratio', 'p_median', 'fft_peak_freq_diff', 'p_mw', 'p_t_welch', 'p_fligner', 'neglog10_p_mean', 'p_bartlett', 'kurt_diff', 'fft_peak_power_ratio', 'len_a', 'len_b', 'std_ratio', 'skew_diff', 'p_levene', 'abs_mean_diff', 'len_ratio', 'acf3_diff', 'acf4_diff', 'cvmd_stat', 'pct10_diff', '

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

F = pd.read_parquet("features.parquet")
print(F.shape)
X = F.values
y = y_train.astype(int).to_numpy()
# Define the parameter grid for MLPClassifier
mlp_param_grid = {
    "clf__hidden_layer_sizes": [(128,), (128, 32), (64, 32),(64,)],
    "clf__activation": ["relu"],
    "clf__alpha": [0.0001, 0.001, 0.01],  # L2 regularization
    #"clf__learning_rate": ["constant", "adaptive"],
    "clf__max_iter": [500, 1000],  # Set a reasonable max_iter
}
#{'clf__activation': 'relu', 'clf__alpha': 0.01, 'clf__hidden_layer_sizes': (128,), 
# #'clf__learning_rate': 'constant', 'clf__max_iter': 500}

# Define the pipeline for MLPClassifier
mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(random_state=42))
])

# Perform grid search
grid_search = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=mlp_param_grid,
    scoring="roc_auc",  # Use AUC as the scoring metric
    cv=3,  # 3-fold cross-validation
    verbose=2,
    n_jobs=-1  # Use all available cores
)

# Fit the grid search on the training data
grid_search.fit(X, y)

# Get the best estimator and parameters
best_mlp = grid_search.best_estimator_
print("Best parameters for MLP:", grid_search.best_params_)


(10001, 48)
Fitting 3 folds for each of 24 candidates, totalling 72 fits


/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(128,), clf__max_iter=500; total time=  39.7s
[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(128, 32), clf__max_iter=500; total time=  39.3s
[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(128, 32), clf__max_iter=500; total time=  39.9s
[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(128, 32), clf__max_iter=1000; total time=  40.1s
[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(128,), clf__max_iter=1000; total time=  42.6s
[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(128, 32), clf__max_iter=500; total time=  45.4s
[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(128,), clf__max_iter=500; total time=  46.0s


/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(64, 32), clf__max_iter=500; total time=  45.5s
[CV] END clf__activation=relu, clf__alpha=0.0001, clf__hidden_layer_sizes=(128, 32), clf__max_iter=1000; total time=  47.5s


/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


... (30 outputs left out)

[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64,), clf__max_iter=500; total time=  23.8s
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(128,), clf__max_iter=1000; total time= 1.4min
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64, 32), clf__max_iter=1000; total time=  38.4s
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(128,), clf__max_iter=1000; total time= 1.4min
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64, 32), clf__max_iter=500; total time=  44.8s


/home/endrem/project/structural_break/competitions/competitions/structural-break/my_break/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64,), clf__max_iter=500; total time=  28.7s
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64,), clf__max_iter=1000; total time=  19.3s
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64, 32), clf__max_iter=1000; total time=  37.9s
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64,), clf__max_iter=1000; total time=  27.4s
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64, 32), clf__max_iter=1000; total time=  51.3s
[CV] END clf__activation=relu, clf__alpha=0.01, clf__hidden_layer_sizes=(64,), clf__max_iter=1000; total time=  32.5s
Best parameters for MLP: {'clf__activation': 'relu', 'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (64,), 'clf__max_iter': 500}


NameError: name 'base_clfs' is not defined

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for RandomForestClassifier
rf_param_grid = {
    "n_estimators": [100, 200, 300],  # Number of trees
    "max_depth": [10, 20, 30, None],  # Maximum depth of the trees
    "min_samples_split": [2, 5, 10],  # Minimum samples required to split a node
    "min_samples_leaf": [1, 2, 4],    # Minimum samples required at a leaf node
    "class_weight": ["balanced", "balanced_subsample"],  # Handle class imbalance
}

# Define the RandomForestClassifier
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Perform grid search
grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    scoring="roc_auc",  # Use AUC as the scoring metric
    cv=3,  # 3-fold cross-validation
    verbose=2,
    n_jobs=-1  # Use all available cores
)

# Fit the grid search on the training data
grid_search_rf.fit(X, y)

# Get the best estimator and parameters
best_rf = grid_search_rf.best_estimator_
print("Best parameters for Random Forest:", grid_search_rf.best_params_)

# Replace the RandomForestClassifier in base_clfs with the tuned model
base_clfs["rf"] = best_rf

Fitting 3 folds for each of 216 candidates, totalling 648 fits
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   3.6s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   4.1s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   4.5s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   6.2s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   7.4s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   8.1s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   3.7s
[CV] END class_weight=balanced, max_depth=10, min_samples_leaf=1, min

NameError: name 'base_clfs' is not defined